# Fase 1: EDA y limpieza inicial

Introducción: esta primera fase tiene como objetivo validar la calidad del catálogo integrado de StreamView, unir los datos de películas y series desde `data/raw` y preparar una base analítica limpia para la siguiente etapa de ingeniería de variables y modelado.

## Objetivos
- Configurar el entorno de trabajo.
- Cargar y unir los orígenes de datos desde `data/raw`.
- Revisar estructura, tipos, duplicados y valores faltantes.
- Normalizar y limpiar el catálogo integrado.
- Generar análisis descriptivo y guardar gráficos en `images/`.
- Exportar la versión procesada para fases posteriores.

Conclusión: si esta etapa queda consistente, la base analítica será sólida y reproducible para el resto del proyecto.


In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("No se encontró la estructura data/raw del proyecto.")

project_root = find_project_root(Path.cwd())
raw_dir = project_root / "data" / "raw"
processed_dir = project_root / "data" / "processed"
images_dir = project_root / "images"
processed_dir.mkdir(parents=True, exist_ok=True)
images_dir.mkdir(parents=True, exist_ok=True)

movies_path = raw_dir / "netflix_movies_detailed_up_to_2025.csv"
tv_shows_path = raw_dir / "netflix_tv_shows_detailed_up_to_2025.csv"
output_path = processed_dir / "catalogo_streamview.csv"

print(f"Proyecto: {project_root}")
print(f"Raw: {raw_dir}")
print(f"Processed: {processed_dir}")
print(f"Images: {images_dir}")
print("Archivos encontrados:")
for file in sorted(raw_dir.glob("*.csv")):
    print(f" - {file.name}")


In [ ]:
movies = pd.read_csv(movies_path, low_memory=False)
tv_shows = pd.read_csv(tv_shows_path, low_memory=False)

print(f"Películas cargadas: {movies.shape}")
print(f"Series cargadas: {tv_shows.shape}")
print("\nVista previa de películas:")
display(movies.head(3))
print("\nVista previa de series:")
display(tv_shows.head(3))


## Sección 1: estructura inicial y calidad de entradas

Introducción: comenzamos revisando la estructura general de ambos orígenes para detectar incoherencias en columnas, tipos y calidad antes de integrarlos.

Conclusión: las dos fuentes comparten el mismo dominio, pero necesitan homogeneización de columnas y tipos para poder fusionarse de forma segura.


In [ ]:
print("Tipos de columnas en películas:")
print(movies.dtypes.head(20).to_string())
print("\nTipos de columnas en series:")
print(tv_shows.dtypes.head(20).to_string())

print("\nColumnas de películas:")
print(movies.columns.tolist())
print("\nColumnas de series:")
print(tv_shows.columns.tolist())

print("\nValores faltantes por dataset:")
for name, df in [("Movies", movies), ("TV Shows", tv_shows)]:
    nulls = df.isna().sum().sort_values(ascending=False)
    nulls = nulls[nulls > 0]
    print(f"\n{name}:")
    print(nulls.head(10).to_string())

print("\nDuplicados por dataset:")
print(f"Movies duplicadas: {movies.duplicated().sum()}")
print(f"TV Shows duplicadas: {tv_shows.duplicated().sum()}")


## Sección 2: limpieza, normalización e integración

Introducción: en esta sección se homogeneizan nombres de columnas, se corrigen vacíos, se estandarizan tipos y se concatena el catálogo para obtener una base analítica única.

Conclusión: al cerrar esta etapa ya queda consolidado un dataset integrado preparado para análisis exploratorio y visualización.


In [ ]:
MULTIVALUE_COLUMNS = ["cast", "director", "country", "genres"]

def normalize_multivalue(value):
    if pd.isna(value):
        return np.nan
    tokens = [part.strip() for part in str(value).split(",") if part.strip()]
    tokens = [re.sub(r"\s+", " ", token).strip() for token in tokens]
    normalized = {}
    for token in tokens:
        normalized[token.casefold()] = token.strip().title()
    canonical = sorted(normalized.values(), key=lambda x: x.casefold())
    return ", ".join(canonical) if canonical else np.nan

def clean_source(dataframe):
    cleaned = dataframe.copy()
    for column in cleaned.columns:
        if cleaned[column].dtype == "object":
            cleaned[column] = cleaned[column].astype(str).str.strip()
            cleaned[column] = cleaned[column].replace(["", "nan", "NaN", "N/A", "n/a", "None", "none"], np.nan)
    for column in MULTIVALUE_COLUMNS:
        if column in cleaned.columns:
            cleaned[column] = cleaned[column].map(normalize_multivalue).astype(object)
    cleaned = cleaned.drop_duplicates().reset_index(drop=True)
    return cleaned

movies_clean = clean_source(movies)
tv_shows_clean = clean_source(tv_shows)

for df in [movies_clean, tv_shows_clean]:
    df.columns = [str(col).strip().lower().replace(" ", "_") for col in df.columns]
    for col in df.columns:
        if df[col].dtype == "object":
            df[col] = df[col].replace(["", "nan", "NaN", "N/A", "n/a", "None", "none"], np.nan)

movies_clean["type"] = "Movie"
tv_shows_clean["type"] = "TV Show"

for df in [movies_clean, tv_shows_clean]:
    if "release_year" in df.columns:
        df["release_year"] = pd.to_numeric(df["release_year"], errors="coerce").astype("Int64")
    if "runtime" in df.columns:
        df["runtime"] = pd.to_numeric(df["runtime"], errors="coerce")
    if "budget" in df.columns:
        df["budget"] = pd.to_numeric(df["budget"], errors="coerce")
    if "revenue" in df.columns:
        df["revenue"] = pd.to_numeric(df["revenue"], errors="coerce")

column_order = movies_clean.columns.tolist()
tv_shows_clean = tv_shows_clean.reindex(columns=column_order)
catalogo = pd.concat([movies_clean, tv_shows_clean], ignore_index=True)
catalogo = catalogo.drop(columns=["description", "show_id", "duration"], errors="ignore")

print(f"Movies limpias: {movies_clean.shape}")
print(f"TV Shows limpias: {tv_shows_clean.shape}")
print(f"Catálogo integrado: {catalogo.shape}")
display(catalogo.head(3))


## Sección 3: información del dataframe y distribuciones categóricas

Introducción: después de la limpieza y la unión, revisamos el dataframe final para confirmar su estructura, su densidad de información y la distribución de las variables clave para negocio antes de construir el score.

Conclusión: con esta revisión se valida que la base sea consistente para la ingeniería de variables y para la construcción del modelo predictivo.


In [ ]:
catalogo.info()
print("\nResumen por tipo de contenido:")
print(catalogo["type"].value_counts().to_string())

for column in ["country", "language", "rating", "genres"]:
    if column in catalogo.columns:
        top_values = catalogo[column].dropna().astype(str).value_counts().head(8)
        print(f"\nTop valores de {column}:")
        print(top_values.to_string())

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
for ax, column in zip(axes.flatten(), ["type", "rating", "country", "language"]):
    if column in catalogo.columns:
        top_values = catalogo[column].dropna().astype(str).value_counts().head(8)
        top_values.plot(kind="bar", ax=ax, color="#4c72b0")
        ax.set_title(f"Top 8 de {column}")
        ax.set_xlabel(column)
        ax.set_ylabel("Frecuencia")
        ax.tick_params(axis="x", rotation=45)

fig.tight_layout()
fig.savefig(images_dir / "distribuciones_categoricas.png", dpi=200, bbox_inches="tight")
plt.close(fig)
print("\nGráfico categórico guardado: distribuciones_categoricas.png")


## Sección 4: análisis cuantitativo y distribuciones

Introducción: una vez validada la estructura y la calidad del dataset, revisamos las distribuciones numéricas más relevantes para detectar señales de volumen, valoración y comportamiento del catálogo.

Conclusión: estas distribuciones orientan la definición de las variables candidatas para el score y para la etapa de modelado predictivo.


## Sección 4: relación entre vote_count y vote_average

Introducción: revisamos la relación entre visibilidad y valoración para comprobar si los títulos con más respaldo tienen realmente mejor percepción, o si hay señales de sesgo por pocos votos.

Conclusión: esta comparación ayuda a definir la calidad del score y evita decisiones guiadas por visibilidad aislada o por pocas evaluaciones.


In [ ]:
if {"vote_count", "vote_average"}.issubset(catalogo.columns):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(catalogo["vote_count"].dropna(), catalogo["vote_average"].dropna(), alpha=0.4, s=25)
    ax.set_title("Relación entre vote_count y vote_average")
    ax.set_xlabel("vote_count")
    ax.set_ylabel("vote_average")
    fig.tight_layout()
    fig.savefig(images_dir / "scatter_vote_count_vote_average.png", dpi=200, bbox_inches="tight")
    plt.close(fig)
    print("Gráfico guardado: scatter_vote_count_vote_average.png")

    corr = catalogo[["vote_count", "vote_average", "popularity", "release_year"]].corr(numeric_only=True)
    print("\nMatriz de correlación:")
    print(corr.round(3).to_string())

    corr_fig, corr_ax = plt.subplots(figsize=(8, 6))
    corr_map = corr_ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    corr_ax.set_title("Heatmap de correlación")
    corr_ax.set_xticks(range(len(corr.columns)))
    corr_ax.set_yticks(range(len(corr.index)))
    corr_ax.set_xticklabels(corr.columns, rotation=45)
    corr_ax.set_yticklabels(corr.index)
    for i in range(len(corr.columns)):
        for j in range(len(corr.index)):
            corr_ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", color="black")
    corr_fig.colorbar(corr_map, ax=corr_ax, fraction=0.046, pad=0.04)
    corr_fig.tight_layout()
    corr_fig.savefig(images_dir / "heatmap_correlacion.png", dpi=200, bbox_inches="tight")
    plt.close(corr_fig)
    print("Gráfico guardado: heatmap_correlacion.png")
else:
    print("Las columnas vote_count y vote_average no están disponibles en el dataset final.")


In [ ]:
summary = catalogo.describe(include="all").transpose()
print(summary.head(25).to_string())

print("\nConteo por tipo de contenido:")
print(catalogo["type"].value_counts().to_string())

for column in ["release_year", "vote_average", "popularity", "runtime"]:
    if column in catalogo.columns:
        fig, ax = plt.subplots(figsize=(10, 5))
        catalogo[column].dropna().hist(bins=30, edgecolor="black", ax=ax)
        ax.set_title(f"Distribución de {column}")
        ax.set_xlabel(column)
        ax.set_ylabel("Frecuencia")
        fig.tight_layout()
        fig.savefig(images_dir / f"distribucion_{column}.png", dpi=200, bbox_inches="tight")
        plt.close(fig)
        print(f"Gráfico guardado: distribucion_{column}.png")

fig, ax = plt.subplots(figsize=(8, 5))
conteo_tipo = catalogo["type"].value_counts()
conteo_tipo.plot(kind="bar", ax=ax, color=["#4c72b0", "#55a868"])
ax.set_title("Conteo por tipo de contenido")
ax.set_xlabel("Tipo")
ax.set_ylabel("Número de registros")
ax.set_xticklabels(["Movie", "TV Show"], rotation=0)
fig.tight_layout()
fig.savefig(images_dir / "conteo_tipo_contenido.png", dpi=200, bbox_inches="tight")
plt.close(fig)

print(f"\nGráficos guardados en: {images_dir}")


## Sección 5: exportación final y cierre

Introducción: la base ya fue revisada, normalizada, integrada y analizada visualmente. En este punto se exporta la versión procesada y se deja evidencia gráfica para la siguiente fase del proyecto.

Conclusión: la fase 1 termina con un catálogo integrado, validado y listo para la ingeniería de variables, el modelado predictivo y la presentación de resultados.


In [ ]:
candidate_vars = [
    "type",
    "genres",
    "country",
    "language",
    "release_year",
    "popularity",
    "vote_count",
    "vote_average",
    "budget",
    "revenue",
    "rating",
]

print("Variables candidatas para la fase 2:")
for variable in candidate_vars:
    if variable in catalogo.columns:
        print(f" - {variable}")

print("\nVariables financieras disponibles en películas:")
movie_financial = catalogo[catalogo["type"] == "Movie"][ ["budget", "revenue"] ]
print(movie_financial.describe().to_string())

catalogo.to_csv(output_path, index=False, na_rep="NaN")

print(f"Archivo exportado: {output_path}")
print("Archivos guardados en images:")
for file in sorted(images_dir.glob("*.png")):
    print(f" - {file.name}")

print("\nResumen final:")
print(catalogo.shape)
print(catalogo.dtypes.head(20).to_string())
